In [1]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [2]:
import folium
import pandas as pd

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

In [5]:
import pandas as pd
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'

resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())

spacex_df = pd.read_csv(spacex_csv_file)

In [6]:
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [7]:
import folium

center_lat = spacex_df['Lat'].mean()
center_lon = spacex_df['Long'].mean()

spacex_map = folium.Map(location=[center_lat, center_lon], zoom_start=5)

for index, row in spacex_df.iterrows():
    folium.Marker(
        location=[row['Lat'], row['Long']],
        popup=row['Launch Site']
    ).add_to(spacex_map)

spacex_map

In [8]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [9]:
import folium

center_lat = launch_sites_df['Lat'].mean()
center_lon = launch_sites_df['Long'].mean()

site_map = folium.Map(location=[center_lat, center_lon], zoom_start=4)

In [10]:
for index, row in launch_sites_df.iterrows():
    folium.Marker(
        location=[row['Lat'], row['Long']],
        popup=row['Launch Site'],
        tooltip=row['Launch Site']
    ).add_to(site_map)

In [11]:
site_map

In [12]:
folium.Marker(
    location=[row['Lat'], row['Long']],
    popup=row['Launch Site'],
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(site_map)

In [14]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [15]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [16]:
import folium
from folium.features import DivIcon

# Initial map
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add circles + markers for each launch site
for index, row in launch_sites_df.iterrows():
    
    # Circle
    folium.Circle(
        location=[row['Lat'], row['Long']],
        radius=1000,
        color='#d35400',
        fill=True
    ).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    
    # Text label
    folium.Marker(
        location=[row['Lat'], row['Long']],
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12; color:#d35400;"><b>{row["Launch Site"]}</b></div>'
        )
    ).add_to(site_map)

# Display map
site_map

#### Are all launch sites near the Equator?
##### NO

#### Are all launch sites close to the coast?
##### YES

👉 Launch sites are not near the equator, but are located in relatively lower latitudes (Florida) to benefit from Earth’s rotational speed.

👉 All launch sites are very close to coastlines because:

It ensures safe rocket launches over the ocean
Minimizes risk to human life and infrastructure
Supports optimal launch trajectories

In [18]:
# Create color column based on class
spacex_df['marker_color'] = spacex_df['class'].map({1: 'green', 0: 'red'})

In [19]:
from folium.plugins import MarkerCluster

marker_cluster = MarkerCluster()

In [20]:
site_map.add_child(marker_cluster)

In [21]:
for index, row in spacex_df.iterrows():
    
    marker = folium.Marker(
        location=[row['Lat'], row['Long']],
        icon=folium.Icon(
            color=row['marker_color']
        ),
        popup=f"Launch Site: {row['Launch Site']}<br>Outcome: {'Success' if row['class']==1 else 'Failure'}"
    )
    
    marker_cluster.add_child(marker)

In [22]:
site_map

In [23]:
launch_site_lat = 28.562302
launch_site_lon = -80.577356

In [24]:
coast_lat = 28.56367
coast_lon = -80.57163

In [26]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # Earth radius in km

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [27]:
distance_coast = calculate_distance(
    launch_site_lat, launch_site_lon,
    coast_lat, coast_lon
)

In [28]:
print(calculate_distance)

<function calculate_distance at 0x5f58b30>


In [29]:
distance_coast = calculate_distance(
    launch_site_lat, launch_site_lon,
    coast_lat, coast_lon
)

print(distance_coast)

0.5797120571581803


In [30]:
from folium.features import DivIcon

distance_marker = folium.Marker(
    location=[coast_lat, coast_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html=f'<div style="font-size: 12; color:#d35400;"><b>{distance_coast:.2f} KM</b></div>'
    )
)

site_map.add_child(distance_marker)

In [31]:
lines = folium.PolyLine(
    locations=[
        [launch_site_lat, launch_site_lon],
        [coast_lat, coast_lon]
    ],
    weight=2,
    color='blue'
)

site_map.add_child(lines)

In [32]:
site_map

In [33]:
# Launch site (same as before)
launch_site_lat = 28.562302
launch_site_lon = -80.577356

# Nearby city point (approx)
city_lat = 28.3922
city_lon = -80.6077

In [34]:
distance_city = calculate_distance(
    launch_site_lat, launch_site_lon,
    city_lat, city_lon
)

In [35]:
from folium.features import DivIcon

city_marker = folium.Marker(
    location=[city_lat, city_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html=f'<div style="font-size: 12; color:blue;"><b>{distance_city:.2f} KM</b></div>'
    )
)

site_map.add_child(city_marker)

In [36]:
city_line = folium.PolyLine(
    locations=[
        [launch_site_lat, launch_site_lon],
        [city_lat, city_lon]
    ],
    weight=2,
    color='blue'
)

site_map.add_child(city_line)

In [38]:
rail_lat = 28.5720
rail_lon = -80.5850

In [40]:
distance_rail = calculate_distance(
    launch_site_lat, launch_site_lon,
    rail_lat, rail_lon
)

In [41]:
rail_marker = folium.Marker(
    location=[rail_lat, rail_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html=f'<div style="font-size: 12; color:green;"><b>{distance_rail:.2f} KM</b></div>'
    )
)

site_map.add_child(rail_marker)

In [42]:
rail_line = folium.PolyLine(
    locations=[
        [launch_site_lat, launch_site_lon],
        [rail_lat, rail_lon]
    ],
    color='green'
)

site_map.add_child(rail_line)

✔ Near highways?

👉 YES → for transport/logistics

✔ Near railways?

👉 YES → heavy equipment movement

✔ Near cities?

👉 Not too close → safety reasons

##### Launch sites are strategically located near transportation infrastructure such as highways and railways to facilitate logistics and movement of materials. However, they maintain a safe distance from major cities to minimize risks. Additionally, their proximity to coastlines ensures safe launch trajectories over water.